# Lab 28 — A2A endpoint from scratch

> ⏱ 90-110 min · 🟡 Intermediate

Build a working A2A endpoint with the official Python SDK (`a2a-sdk` 1.0.3). Serve its Agent Card at `/.well-known/agent-card.json`. Handle a real JSON-RPC `SendMessage` request. Observe the full Task lifecycle (`submitted` → `working` → `completed`) end-to-end with an actual HTTP roundtrip.

**Prerequisites**: [A2A foundations](../../concepts/tools/a2a-foundations.md). Helpful: [Lab 25 (MCP server from scratch)](../25-mcp-server-from-scratch/) for the analogous "build the server side" pattern one protocol layer down.

**Strategy**: the notebook drives a `subprocess.Popen`-launched uvicorn server (not an in-Jupyter event-loop server — that's fragile). Each test cell sends a real HTTP request to the subprocess and parses the protobuf-shaped JSON response. Final cell terminates the subprocess cleanly.

## Step 0 — Environment setup

Verify `a2a-sdk` is installed (the protobuf-based SDK 1.0.3) plus `httpx` for the client and `uvicorn` for the server.

In [ ]:
import sys
import subprocess
import time
import json
from pathlib import Path

LAB_DIR = Path.cwd()
print(f"Lab working directory: {LAB_DIR}")

try:
    import a2a
    print(f"a2a-sdk: imported (version: {getattr(a2a, '__version__', 'introspect via pip')})")
except ImportError:
    print("✗ a2a-sdk not installed. Run: pip install 'a2a-sdk>=1.0,<2.0'")
    sys.exit(1)

try:
    import httpx
    print(f"httpx: {httpx.__version__}")
except ImportError:
    print("✗ httpx not installed. Run: pip install httpx")
    sys.exit(1)

try:
    import uvicorn
    print(f"uvicorn: {uvicorn.__version__}")
except ImportError:
    print("✗ uvicorn not installed. Run: pip install uvicorn")
    sys.exit(1)

# No API keys required for this lab.
print()
print("✓ Environment ready. No external API key needed.")

## Step 1 — Inspect the SDK 1.0.3 surface

The protobuf-based types are a 2026 shift from earlier Pydantic-based versions. Tutorials and blog posts dated pre-2026 will show patterns that no longer work. Step 1 makes the new shape explicit before we use it.

In [ ]:
from a2a.types import (
    AgentCard, AgentSkill, AgentCapabilities, AgentInterface,
    Message, Task, Role, TaskState, Part,
)

print("=== Protobuf vs Pydantic ===")
print(f"AgentCard type: {type(AgentCard).__name__}")
print(f"  Is protobuf: {'a2a_pb2' in str(AgentCard.__module__)}")
print(f"  Has model_fields? {hasattr(AgentCard, 'model_fields')}")
print(f"  Has DESCRIPTOR? {hasattr(AgentCard, 'DESCRIPTOR')}")
print()

print("=== AgentCard fields (from protobuf DESCRIPTOR) ===")
for field in AgentCard.DESCRIPTOR.fields:
    print(f"  {field.name}")
print()

print("=== Role enum values ===")
for value in Role.DESCRIPTOR.values:
    print(f"  Role.{value.name} = {value.number}")
print()

print("=== TaskState enum values ===")
for value in TaskState.DESCRIPTOR.values:
    print(f"  TaskState.{value.name} = {value.number}")

**Key observations:**

- Protobuf types use `snake_case` field names on the wire (`message_id`, `task_id`, `context_id`, `protocol_binding`, `protocol_version`). When you see `messageId`/`taskId` in JSON responses, that's the SDK's JSON marshaling adding camelCase for the wire format.
- Enums are integer-valued with explicit `*_UNSPECIFIED = 0` (`Role.ROLE_USER = 1`, `Role.ROLE_AGENT = 2`).
- `TaskState` has 9 values: 1 unspecified + 4 non-terminal (`SUBMITTED`, `WORKING`, `INPUT_REQUIRED`, `AUTH_REQUIRED`) + 4 terminal (`COMPLETED`, `FAILED`, `CANCELED`, `REJECTED`).

## Step 2 — Build the Agent Card

The Agent Card is the JSON document the server publishes at `/.well-known/agent-card.json` — the "OpenAPI for Agents." Per Module 5, it advertises what the agent can do, what auth it requires, and where to reach it.

We'll author a minimal card with one skill: `greet`. The card has no auth requirements (`SecurityRequirement` left empty), no streaming, no push notifications — the simplest possible production-shaped card.

In [ ]:
# Build the components

skill = AgentSkill(
    id="greet",
    name="Greeting",
    description="Echo a greeting back to the user",
    tags=["hello"],
    input_modes=["text/plain"],
    output_modes=["text/plain"],
)

capabilities = AgentCapabilities(
    streaming=False,
    push_notifications=False,
)

# AgentInterface declares how the agent can be reached
# Field name is `protocol_binding` (not `transport` — that's a common mistake)
interface = AgentInterface(
    protocol_binding="JSONRPC",
    url="http://127.0.0.1:9999",
)

agent_card = AgentCard(
    name="hello-agent",
    description="A trivial A2A agent that echoes greetings",
    version="0.1.0",
    capabilities=capabilities,
    supported_interfaces=[interface],
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[skill],
)

# Serialize to JSON (using protobuf's MessageToDict for inspection)
from google.protobuf.json_format import MessageToDict
card_dict = MessageToDict(agent_card)
print(json.dumps(card_dict, indent=2))

## Step 3 — Implement the AgentExecutor

The `AgentExecutor` is the abstract base class your agent subclasses. Two methods to implement:

- `execute(context, event_queue)` — handle an incoming task; publish lifecycle events
- `cancel(context, event_queue)` — handle a cancellation request

**Critical pattern (the gotcha):** before calling any `TaskUpdater` lifecycle method, you must enqueue the initial `Task` object. Calling `await updater.submit()` first yields `-32006 INVALID_AGENT_RESPONSE` ("Agent should enqueue Task before TaskStatusUpdateEvent event"). The canonical fix is `new_task_from_user_message(context.message)` → `event_queue.enqueue_event(task)` → then the TaskUpdater calls.

We'll write the server module to a scratch file so we can run it via subprocess in Step 5.

In [ ]:
SERVER_PATH = LAB_DIR / "hello_agent_server.py"

SERVER_CODE = """\
\"\"\"Lab 28 A2A server — hello-agent built on a2a-sdk 1.0.3.\"\"\"
from a2a.types import (
    AgentCard, AgentSkill, AgentCapabilities, AgentInterface,
)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.helpers import new_text_part, new_task_from_user_message
from starlette.applications import Starlette
import uvicorn


class HelloAgent(AgentExecutor):
    \"\"\"Trivial A2A agent: extracts text from incoming Message, echoes back.\"\"\"

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        # 1. Extract text from the incoming message's parts
        message_text = \"\"
        if context.message and context.message.parts:
            for part in context.message.parts:
                if part.HasField(\"text\"):
                    message_text += part.text

        # 2. Enqueue the initial Task object FIRST (critical — see notebook gotcha)
        task = new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)

        # 3. Use TaskUpdater to publish the lifecycle: working → artifact → completed
        #    (submit() is implicit since the Task was enqueued with SUBMITTED state)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()
        response = f\"Hello! You said: {message_text}\"
        await updater.add_artifact(parts=[new_text_part(response)], name=\"response\")
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        raise NotImplementedError(\"Cancel not implemented for the toy hello-agent\")


# Agent Card (same shape as Step 2 of the notebook)
agent_card = AgentCard(
    name=\"hello-agent\",
    description=\"A trivial A2A agent that echoes greetings\",
    version=\"0.1.0\",
    capabilities=AgentCapabilities(streaming=False, push_notifications=False),
    supported_interfaces=[
        AgentInterface(protocol_binding=\"JSONRPC\", url=\"http://127.0.0.1:9999\")
    ],
    default_input_modes=[\"text/plain\"],
    default_output_modes=[\"text/plain\"],
    skills=[
        AgentSkill(
            id=\"greet\",
            name=\"Greeting\",
            description=\"Echo a greeting back to the user\",
            tags=[\"hello\"],
            input_modes=[\"text/plain\"],
            output_modes=[\"text/plain\"],
        )
    ],
)


def build_app():
    \"\"\"Wire the route factory; mount on Starlette.\"\"\"
    handler = DefaultRequestHandler(
        agent_executor=HelloAgent(),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )
    card_routes = create_agent_card_routes(agent_card=agent_card)
    rpc_routes = create_jsonrpc_routes(request_handler=handler, rpc_url=\"/\")
    return Starlette(routes=card_routes + rpc_routes)


if __name__ == \"__main__\":
    uvicorn.run(build_app(), host=\"127.0.0.1\", port=9999, log_level=\"warning\")
"""

SERVER_PATH.write_text(SERVER_CODE)
print(f"Wrote {SERVER_PATH} ({SERVER_PATH.stat().st_size} bytes)")

## Step 4 — Wire the routes

The SDK 1.0 changed how servers are wired. Pre-1.0 used an `A2AStarletteApplication` wrapper class; 1.0 replaced it with a **route factory** pattern that returns Starlette `Route` lists you mount yourself.

In `hello_agent_server.py` (just written) the wiring is:

```python
handler = DefaultRequestHandler(
    agent_executor=HelloAgent(),
    task_store=InMemoryTaskStore(),
    agent_card=agent_card,  # required in 1.0; the handler verifies declared capabilities
)
card_routes = create_agent_card_routes(agent_card=agent_card)
rpc_routes = create_jsonrpc_routes(request_handler=handler, rpc_url="/")
app = Starlette(routes=card_routes + rpc_routes)
```

Three things to note:

1. **`DefaultRequestHandler` requires `agent_card`** (was optional in pre-1.0). The handler uses it to verify the agent's declared capabilities — e.g. it checks whether streaming is supported before handling `SendStreamingMessage` requests.
2. **`create_jsonrpc_routes` requires `rpc_url`** (was implicit `/` in pre-1.0). The explicit URL lets you mount A2A at any path — e.g. `rpc_url="/api/v1/a2a"` if you're sharing the Starlette app with other handlers.
3. **The route factory returns `list[Route]`** that compose with any existing Starlette/FastAPI routing. This makes A2A endpoints embeddable in larger apps.

Now let's verify the file imports cleanly:

In [ ]:
# Import the just-written server module to verify it parses + builds
import importlib.util
spec = importlib.util.spec_from_file_location("hello_agent_server", str(SERVER_PATH))
server_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server_module)

# Build the app (without running uvicorn)
app = server_module.build_app()
print(f"App type: {type(app).__name__}")
print()
print("Routes registered:")
for route in app.routes:
    print(f"  {route.path}")

## Step 5 — Run the server in a subprocess

Running uvicorn inside the Jupyter event loop is fragile (it tries to start a new event loop inside the existing one). `subprocess.Popen` is the right shape: the server runs in its own Python process; the notebook drives it via HTTP; cleanup is `process.terminate()` away.

The cell launches uvicorn, waits briefly for it to start, then health-probes the Agent Card URL to confirm it's up. If port 9999 is in use, you'll see a clear `ConnectError`; either kill the existing process or change the port in the scratch file.

In [ ]:
server_process = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give uvicorn a moment to bind to the port and start serving
time.sleep(1.5)

# Health probe
health_ok = False
for _attempt in range(5):
    try:
        r = httpx.get("http://127.0.0.1:9999/.well-known/agent-card.json", timeout=2.0)
        if r.status_code == 200:
            health_ok = True
            break
    except (httpx.ConnectError, httpx.ReadTimeout):
        time.sleep(0.5)

if not health_ok:
    # Capture any stderr from the server for debugging
    if server_process.poll() is not None:
        print("✗ Server process exited prematurely. Stderr:")
        print(server_process.stderr.read().decode()[:2000])
    else:
        print("✗ Server is running but not responding to health probe")
    raise RuntimeError("Server failed to start")

print(f"✓ Server running at http://127.0.0.1:9999 (PID {server_process.pid})")

## Step 6 — Probe the Agent Card

The Agent Card is the entry point for any A2A interaction. A client agent fetching the card learns *what* the remote agent can do, *how* to reach it (URL + transport), and *what auth* it requires. Without the card, there's no protocol — the card *is* the discovery mechanism.

Real A2A clients fetch the card once at startup and cache it (with periodic refresh for change detection — analogous to MCP's tool-description fingerprinting in Lab 27). For this lab we just fetch it once and inspect.

In [ ]:
card_response = httpx.get(
    "http://127.0.0.1:9999/.well-known/agent-card.json",
    timeout=5.0,
)
print(f"Status: {card_response.status_code}")
card = card_response.json()
print(json.dumps(card, indent=2))

**Observations about the card the server serves:**

- `protocolVersion: "0.3"` — the SDK 1.0.3 still defaults to advertising 0.3 in the card for backward compatibility with v0.3 clients, even though the server expects v1.0 requests by default. This is a quirk of the current SDK release; it should converge in future versions.
- Field naming is camelCase in JSON (`supportedInterfaces`, `defaultInputModes`, `protocolBinding`) — the SDK's protobuf-to-JSON marshaling adds the casing. Inside Python you use `snake_case` (`supported_interfaces`).
- `preferredTransport: "JSONRPC"` — the wire-level transport name. Other options exist (`GRPC`, `REST`) but JSONRPC is the most-used and the default.
- No `signatures` field — this card is unsigned. v1.0 production deployments add Signed Agent Cards; that's Module 6 territory.

## Step 7 — Send a real JSON-RPC `SendMessage`

The canonical test of an A2A endpoint: client sends a `SendMessage` JSON-RPC request; server creates a Task; agent executes; Task transitions through the lifecycle; final response contains the completed Task with its artifacts.

Two things to get right:

1. **Method name is `SendMessage`** (PascalCase, gRPC-style) — not `message/send` (the v0.3 slash-style). The slash-style works only via the SDK's compat layer.
2. **HTTP header `A2A-Version: 1.0`** is required, or you get `-32009 VERSION_NOT_SUPPORTED`. SDK 1.0.3 servers default to expecting v1.0 messages; the header tells the server which version this client is speaking.

In [ ]:
rpc_request = {
    "jsonrpc": "2.0",
    "id": "req-1",
    "method": "SendMessage",
    "params": {
        "message": {
            "message_id": "msg-1",
            "role": "ROLE_USER",
            "parts": [{"text": "world"}],
        }
    },
}

rpc_response = httpx.post(
    "http://127.0.0.1:9999/",
    headers={
        "Content-Type": "application/json",
        "A2A-Version": "1.0",
    },
    json=rpc_request,
    timeout=10.0,
)

print(f"Status: {rpc_response.status_code}")
print()
print(json.dumps(rpc_response.json(), indent=2))

**What the response shows — the full Task lifecycle in one payload:**

- `task.id` and `task.contextId` — UUIDs the server allocated. The context_id groups related tasks across a conversation.
- `task.status.state: "TASK_STATE_COMPLETED"` — the terminal state. Earlier states (`SUBMITTED`, `WORKING`) happened during execution but the final response surfaces only the latest.
- `task.artifacts[0]` — the actual result. One artifact with `name: "response"` containing one text part. Production agents typically return multiple typed artifacts (text + JSON + URLs).
- `task.history` — the conversation history. Contains the original user message that triggered the task.

This is the same shape any A2A v1.0 endpoint returns. A client agent receiving this response treats the `task.artifacts` as the structured output and the `task.status.state` as the success signal.

## Step 8 — Stretch: protocol-version negotiation + GetTask + cleanup

Three quick demonstrations:

**8a — Missing `A2A-Version` header yields `-32009`.** The error makes clear what the server expects; production clients always send the header.

In [ ]:
# Same request, but no A2A-Version header
bad_response = httpx.post(
    "http://127.0.0.1:9999/",
    headers={"Content-Type": "application/json"},
    json={
        "jsonrpc": "2.0",
        "id": "req-bad",
        "method": "SendMessage",
        "params": {"message": {"message_id": "m", "role": "ROLE_USER",
                               "parts": [{"text": "x"}]}},
    },
    timeout=5.0,
)
result = bad_response.json()
print(f"Error code: {result.get('error', {}).get('code')}")
print(f"Error message: {result.get('error', {}).get('message')}")

**8b — `GetTask` retrieves a task by ID.** The task lives in the server's task store (InMemory for this lab; PostgreSQL/MySQL/SQLite for production — see Lab 29). After the SendMessage roundtrip from Step 7, the task is still retrievable. The SDK 1.0 `GetTaskRequest` proto field is `id` (the v0.3 `name` alias is gone — production code uses `id`).

In [ ]:
# Grab the task ID from Step 7's response
task_id_from_step_7 = rpc_response.json()["result"]["task"]["id"]
print(f"Looking up task: {task_id_from_step_7}")
print()

get_task_response = httpx.post(
    "http://127.0.0.1:9999/",
    headers={"Content-Type": "application/json", "A2A-Version": "1.0"},
    json={
        "jsonrpc": "2.0",
        "id": "req-get",
        "method": "GetTask",
        "params": {"id": task_id_from_step_7},
    },
    timeout=5.0,
)
result = get_task_response.json()
if "error" in result:
    print(f"Error: {result['error']}")
else:
    task = result["result"]
    print(f"Retrieved task: id={task['id']}, state={task['status']['state']}")
    print(f"  contextId: {task['contextId']}")

**8c — Clean shutdown.** Terminate the subprocess. Production deployments use `uvicorn`'s graceful-shutdown signals (SIGTERM); `subprocess.terminate()` sends the same.

In [ ]:
server_process.terminate()
try:
    server_process.wait(timeout=5.0)
    print(f"✓ Server stopped cleanly (exit code: {server_process.returncode})")
except subprocess.TimeoutExpired:
    server_process.kill()
    server_process.wait()
    print("⚠ Server didn't respond to SIGTERM; force-killed")

## What you've built

You now have:

- 📄 `hello_agent_server.py` — a runnable A2A v1.0 server using the official SDK 1.0.3
- A working understanding of the protobuf-based type system (no Pydantic in 1.0+)
- Hands-on familiarity with the route factory pattern (`create_agent_card_routes` + `create_jsonrpc_routes`)
- Direct observation of the Task lifecycle (`submitted` → `working` → `completed`) in a real protocol response
- The subprocess pattern for running A2A servers from notebooks without fighting the event loop
- The "enqueue Task before TaskUpdater" gotcha — the most common AgentExecutor bug
- Knowledge of the `A2A-Version: 1.0` header requirement — the second-most common bug

## What's next

The MCP build-consume-secure trio (Modules 1+2+3+4) took three labs (25, 26, 27). The A2A trio will follow the same shape:

- **Module 6 (future batch)** — Building an A2A endpoint at production depth: Signed Agent Cards, `DatabaseTaskStore` (PostgreSQL), OAuth2 auth, streaming via `SendStreamingMessage`, push notifications, OpenTelemetry tracing
- **Module 7 (future batch)** — MCP + A2A composition: the orchestrator pattern with `A2ACardResolver` + `ClientFactory`; agents using MCP for their own tools while using A2A to coordinate with each other; the canonical hybrid pattern per the [Pattern 12 architecture page](../../patterns/12-a2a-federation.md)

## Test yourself

Take the [A2A foundations quiz](../../quizzes/foundations/a2a-foundations.md) — 8 questions covering Module 5 and this lab.